# Setup môi trường

In [2]:
import os
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras

In [3]:
# Set cố định random seed
np.random.seed(42)
tf.random.set_seed(42)

# Tìm hiểu dữ liệu

In [4]:
df = pd.read_csv("train_data.csv")
df.head(5)

,sentence,sentiment
0,awww that s a bummer you shoulda got david car...,0
1,is upset that he can t update his facebook by ...,0
2,i dived many times for the ball managed to sav...,0
3,my whole body feels itchy and like its on fire,0
4,no it s not behaving at all i m mad why am i h...,0


# Kiểm tra dữ liệu

In [5]:
df.info()
df['sentiment'].value_counts()
df = df.dropna()
X = df['sentence']
y = df['sentiment']

<class 'pandas.DataFrame'>
RangeIndex: 1523975 entries, 0 to 1523974
Data columns (total 2 columns):
 #   Column     Non-Null Count    Dtype
---  ------     --------------    -----
 0   sentence   1523975 non-null  str  
 1   sentiment  1523975 non-null  int64
dtypes: int64(1), str(1)
memory usage: 117.1 MB


# Chia train/test

In [6]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.utils import pad_sequences

In [11]:
# Số lượng từ tối đa và độ dài câu
vocab_size = 10000
max_length = 400
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)
tokenizer = Tokenizer(
    num_words=vocab_size,
    oov_token="<OOV>"
)
tokenizer.fit_on_texts(X_train)
X_train_seqs = tokenizer.texts_to_sequences(X_train)

X_test_seqs = tokenizer.texts_to_sequences(X_test)
X_train_padded = pad_sequences(
    X_train_seqs,
    maxlen=max_length,
    padding='post',
    truncating='post'
)


X_test_padded = pad_sequences(
    X_test_seqs,
    maxlen=max_length,
    padding='post',
    truncating='post'
)

print("Train:", len(X_train_padded))

print("Test:", len(X_test_padded))

Train: 1219180
Test: 304795


In [12]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense

model = Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=16,
        input_length=max_length
    ),

    GlobalAveragePooling1D(),

    Dense(
        16,
        activation='relu'
    ),

    Dense(
        1,
        activation='sigmoid'
    )
])

model.summary()

c:\Users\Admin\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:103: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ ?                      │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [13]:
from tensorflow.keras.callbacks import ModelCheckpoint

checkpoint = ModelCheckpoint(
    filepath='model_epoch_{epoch:02d}.keras',
    save_weights_only=False,
    save_best_only=False,
    monitor='val_loss',
    verbose=1
)

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history = model.fit(
    X_train_padded,
    y_train,
    validation_data=(
        X_test_padded,
        y_test
    ),
    epochs=5,
    callbacks=[checkpoint]
)

Epoch 1/5
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5027 - loss: 0.6932
Epoch 1: saving model to model_epoch_01.keras

Epoch 1: finished saving model to model_epoch_01.keras
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 456s 6ms/step - accuracy: 0.5025 - loss: 0.6932 - val_accuracy: 0.4977 - val_loss: 0.6933
Epoch 2/5
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.5028 - loss: 0.6932
Epoch 2: saving model to model_epoch_02.keras

Epoch 2: finished saving model to model_epoch_02.keras
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 231s 6ms/step - accuracy: 0.5026 - loss: 0.6932 - val_accuracy: 0.4977 - val_loss: 0.6933
Epoch 3/5
38093/38100 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.5028 - loss: 0.6932
Epoch 3: saving model to model_epoch_03.keras

Epoch 3: finished saving model to model_epoch_03.keras
38100/38100 ━━━━━━━━━━━━━━━━━━━━ 191s 5ms/step - accuracy: 0.5026 - loss: 0.6932 - val_accuracy: 0.4977 - val_loss: 0.6933
Epoch 4/5
38087/38100 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - a

# Lưu mô hình

In [14]:
model.save('tweet_sentiment_model.keras')

# Sử dụng mô hình

In [ ]:
test_df = pd.read_csv("test_data.csv")
test_reviews = test_df['sentence'].values
test_labels = test_df['sentiment'].values